# Download UltraFeedback for DPO
Downloads `openbmb/UltraFeedback` and extracts chosen/rejected preference pairs.
Each prompt has 4 completions rated by GPT-4; we take the best and worst as our pair.

Run `tokenize_dpo` → `dpo_ultrafeedback` after.

In [ ]:
!pip install -q datasets

In [ ]:
import os
import json
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
OUT_DIR = os.path.join(SPARKYLLM, "datasets_dpo", "ultrafeedback")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_FILE = os.path.join(OUT_DIR, "ultrafeedback_raw.jsonl")
MIN_SCORE_GAP = 1.0

if os.path.exists(OUT_FILE):
    size_mb = os.path.getsize(OUT_FILE) / 1024 / 1024
    print(f"Already exists: {OUT_FILE} ({size_mb:.1f} MB)")
    print("Delete the file and re-run to re-download.")
else:
    print("Loading openbmb/UltraFeedback from HuggingFace...")
    ds = load_dataset("openbmb/UltraFeedback", split="train")
    print(f"Loaded {len(ds):,} prompts")

    written = 0
    skipped_no_scores = 0
    skipped_small_gap = 0
    skipped_empty = 0

    with open(OUT_FILE, "w", encoding="utf-8") as f:
        for ex in ds:
            prompt = ex["instruction"].strip()
            completions = ex.get("completions", [])

            if not prompt or len(completions) < 2:
                skipped_empty += 1
                continue

            # Extract (response, score) pairs
            # UltraFeedback stores scores as `overall_score` directly on each completion
            scored = []
            for comp in completions:
                response = comp.get("response", "").strip()
                overall = comp.get("overall_score")

                if not response or overall is None:
                    continue

                try:
                    score = float(overall)
                except (ValueError, TypeError):
                    continue

                scored.append((response, score))

            if len(scored) < 2:
                skipped_no_scores += 1
                continue

            # Best and worst by score
            scored.sort(key=lambda x: x[1], reverse=True)
            chosen_text, chosen_score = scored[0]
            rejected_text, rejected_score = scored[-1]

            if chosen_score - rejected_score < MIN_SCORE_GAP:
                skipped_small_gap += 1
                continue

            obj = {
                "prompt": prompt,
                "chosen": chosen_text,
                "rejected": rejected_text,
                "chosen_score": chosen_score,
                "rejected_score": rejected_score,
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
            written += 1

    # Save metadata
    meta = {
        "source": "openbmb/UltraFeedback",
        "num_pairs": written,
        "min_score_gap": MIN_SCORE_GAP,
        "skipped_empty": skipped_empty,
        "skipped_no_scores": skipped_no_scores,
        "skipped_small_gap": skipped_small_gap,
    }
    meta_path = os.path.join(OUT_DIR, "meta.json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    size_mb = os.path.getsize(OUT_FILE) / 1024 / 1024
    print(f"\nDone!")
    print(f"  {written:,} preference pairs written")
    print(f"  Skipped: {skipped_empty:,} empty, {skipped_no_scores:,} no scores, {skipped_small_gap:,} gap < {MIN_SCORE_GAP}")
    print(f"  {size_mb:.1f} MB")
    print(f"  Saved to: {OUT_FILE}")